# 06 · Incertidumbre conformal y robustez espacial

**Pregunta.** ¿El candidato aprobado produce intervalos temporalmente válidos y conserva su desempeño cuando se retienen entidades completas?

**Entradas.** Predicciones fuera de tiempo, feature mart A+B, decisión de modelo y `configs/uncertainty.yaml`.

**Método.** Intervalos conformales de residuo absoluto con calibración móvil estrictamente anterior al target y cuantiles por clase A/B cuando hay muestra suficiente. Como prueba espacial separada, cinco grupos deterministas retienen entidades completas tanto del entrenamiento histórico como de la evaluación.

**Criterio de honestidad.** El resultado 2021 sólo puede usar residuos 2018; el resultado 2024 puede usar 2018–2021. No se recalibra con el target que se evalúa. La prueba espacial es diagnóstico de transferencia, no el esquema de entrenamiento productivo.

**Salidas.** Intervalos, cobertura global y por subgrupo, predicciones espaciales, métricas de transferencia, decisión máquina-legible y reporte HTML.

## Entorno, configuración y entradas

Se comprueban las salidas congeladas del notebook 05 y se muestra el contrato antes de calcular resultados.

In [ ]:
from pathlib import Path
import json
import time

import pandas as pd
import yaml
from IPython.display import Markdown, display

from radar27.modeling.uncertainty import (
    build_conformal_intervals,
    run_spatial_entity_holdout,
    write_uncertainty_decision,
)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "configs/uncertainty.yaml").exists():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del proyecto")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RUN_STARTED = time.perf_counter()
with (PROJECT_ROOT / "configs/uncertainty.yaml").open(encoding="utf-8") as stream:
    uncertainty_config = yaml.safe_load(stream)
required_inputs = [
    PROJECT_ROOT / "artifacts/model_decision.json",
    PROJECT_ROOT / "artifacts/tables/feature_mart_ab.parquet",
    PROJECT_ROOT / "artifacts/tables/model_backtest_predictions.parquet",
]
input_checks = {path.relative_to(PROJECT_ROOT).as_posix(): path.exists() for path in required_inputs}
display(pd.Series(input_checks, name="existe").to_frame())
display(pd.Series(uncertainty_config["conformal"], name="valor").to_frame())
assert all(input_checks.values())

## Intervalos conformales móviles

El radio es el cuantil conformal finito de los errores absolutos fuera de tiempo. Se estratifica por clase territorial sólo si el estrato de calibración supera el mínimo configurado; de otro modo se usa el cuantil global y se registra el respaldo.

In [ ]:
intervals, coverage, calibration = build_conformal_intervals(PROJECT_ROOT)
overall_coverage = coverage[coverage.group_dimension.eq("overall")].copy()
display(calibration)
display(overall_coverage)

In [ ]:
conformal_checks = {
    "calibration_strictly_prior": bool(
        (intervals.max_calibration_target_year < intervals.target_year).all()
    ),
    "bounds_are_probabilities": bool(
        intervals.lower.dropna().between(0, 1).all()
        and intervals.upper.dropna().between(0, 1).all()
    ),
    "ordered_intervals": bool((intervals.lower <= intervals.upper).all()),
    "required_diagnostics_present": {
        "overall", "entity_id", "geography_confidence_class",
        "section_size_band", "predicted_risk_decile",
    } <= set(coverage.group_dimension),
}
display(pd.Series(conformal_checks, name="aprobado").to_frame())
assert all(conformal_checks.values())

In [ ]:
coverage_2021 = overall_coverage.set_index("target_year").loc[2021]
coverage_2024 = overall_coverage.set_index("target_year").loc[2024]
display(Markdown(
    f"**Resultado temporal.** Con nominal de 90%, la cobertura observada es "
    f"`{100*coverage_2021.empirical_coverage:.2f}%` en 2021 y "
    f"`{100*coverage_2024.empirical_coverage:.2f}%` en 2024. "
    "La diferencia evidencia cambio entre elecciones; no se corrige usando información del target."
))

## Validación espacial con entidades retenidas

Las 32 entidades se reparten en cinco grupos por orden de ID. Para cada año y grupo, el modelo se reentrena sólo con años anteriores y sin ninguna fila histórica de las entidades evaluadas. Se compara contra el backtest temporal ordinario sobre exactamente las mismas unidades.

In [ ]:
spatial_predictions, spatial_metrics = run_spatial_entity_holdout(PROJECT_ROOT)
overall_spatial = spatial_metrics[spatial_metrics.scope.eq("overall")].copy()
display(overall_spatial)
display(
    spatial_metrics[spatial_metrics.scope.eq("entity_id")]
    .sort_values(["target_year", "weighted_mae_deterioration"], ascending=[True, False])
    .groupby("target_year", as_index=False)
    .head(5)
)

In [ ]:
spatial_checks = {
    "training_years_strictly_prior": bool(
        (spatial_predictions.max_training_target_year < spatial_predictions.target_year).all()
    ),
    "each_entity_is_held_out": bool(spatial_predictions.apply(
        lambda row: str(row.entity_id) in str(row.held_out_entities).split("|"), axis=1
    ).all()),
    "one_prediction_per_unit": not spatial_predictions.duplicated(
        ["target_year", "evaluation_unit_id"]
    ).any(),
    "predictions_are_probabilities": bool(
        spatial_predictions.spatial_holdout_prediction.between(0, 1).all()
    ),
}
display(pd.Series(spatial_checks, name="aprobado").to_frame())
assert all(spatial_checks.values())

## Decisión y reporte

Los umbrales configurados son diagnósticos de preparación, no una forma de seleccionar de nuevo el modelo. Una cobertura temporal insuficiente impide declarar listos los intervalos 2027 aunque el estrés espacial sea favorable.

In [ ]:
uncertainty_decision = write_uncertainty_decision(
    PROJECT_ROOT, coverage, spatial_metrics
)
display(pd.Series(uncertainty_decision).to_frame("valor"))

In [ ]:
report_path = PROJECT_ROOT / "artifacts/reports/uncertainty_spatial_validation.html"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join([
    "<h1>RADAR 27 · Incertidumbre y robustez espacial</h1>",
    "<p>Generado por notebooks/06_uncertainty_spatial_validation.ipynb.</p>",
    "<h2>Calibración</h2>", calibration.to_html(index=False),
    "<h2>Cobertura global</h2>", overall_coverage.to_html(index=False),
    "<h2>Cobertura por subgrupo</h2>", coverage.to_html(index=False),
    "<h2>Robustez espacial global</h2>", overall_spatial.to_html(index=False),
    "<h2>Robustez espacial detallada</h2>", spatial_metrics.to_html(index=False),
    "<h2>Decisión</h2><pre>" + json.dumps(
        uncertainty_decision, ensure_ascii=False, indent=2
    ) + "</pre>",
]), encoding="utf-8")

required_outputs = [
    PROJECT_ROOT / "artifacts/tables/conformal_backtest_intervals.parquet",
    PROJECT_ROOT / "artifacts/tables/conformal_coverage_metrics.csv",
    PROJECT_ROOT / "artifacts/tables/conformal_calibration_summary.csv",
    PROJECT_ROOT / "artifacts/tables/spatial_holdout_predictions.parquet",
    PROJECT_ROOT / "artifacts/tables/spatial_robustness_metrics.csv",
    PROJECT_ROOT / "artifacts/uncertainty_decision.json",
    report_path,
]
final_checks = {
    "all_outputs_exist": all(path.exists() for path in required_outputs),
    **conformal_checks,
    **spatial_checks,
}
display(pd.Series(final_checks, name="aprobado").to_frame())
assert all(final_checks.values())

In [ ]:
readiness = "LISTO" if uncertainty_decision["ready_for_first_2027_projection"] else "NO LISTO"
display(Markdown(
    f"**Cierre: {readiness} para proyectar 2027 con intervalos.** "
    f"La validación espacial pasa: `{uncertainty_decision['spatial_deterioration_diagnostic_passes']}`; "
    f"la cobertura conformal pasa: `{uncertainty_decision['global_coverage_diagnostic_passes']}`. "
    f"Tiempo total: `{time.perf_counter()-RUN_STARTED:.1f}` s."
))